# NARR-to-PRISM Downscaling: Model Inference

This notebook demonstrates how to run inference with a fine-tuned NARR-to-PRISM downscaling model.

We show how to:
1. Load a trained checkpoint
2. Run predictions over the configured inference date range
3. Denormalize outputs to physical units
4. Write results to NetCDF
5. Visualize sample predictions

---

## Setup

Python >= 3.10 is required

Make sure that your current working directory is `granite-wxc/`

In [ ]:
import os
import sys
from pathlib import Path

# Set REPO_ROOT to the granite-wxc repository root
REPO_ROOT = Path(".").resolve()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

NARR_PRISM_DIR = REPO_ROOT / "examples" / "NARR_PRISM"
if str(NARR_PRISM_DIR) not in sys.path:
    sys.path.insert(0, str(NARR_PRISM_DIR))

os.chdir(REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"Working directory: {Path.cwd()}")

In [ ]:
!pip install -q git+https://github.com/NASA-IMPACT/Prithvi-WxC.git

In [ ]:
!pip install -q h5netcdf matplotlib xarray scipy torch tqdm pyyaml

In [ ]:
# Install granitewxc in editable mode from the repo source
!pip install -q -e {REPO_ROOT}

In [ ]:
import logging
import warnings

logging.disable(logging.CRITICAL)
warnings.simplefilter(action="ignore", category=FutureWarning)

---

## Device Configuration

In [ ]:
# ===================== HARDWARE CONFIGURATION (EDIT ME) =====================
# GPUs 2 and 3 are busy with training; use idle GPUs 0 and 1 for parallel date shards.
force_visible_devices = "0,1"  # comma-separated physical GPU ids, or None for all
device_target = "cuda"          # "cuda" or "cpu"
# ============================================================================

if force_visible_devices is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(force_visible_devices)

import torch
import numpy as np

if device_target == "cuda" and not torch.cuda.is_available():
    print("CUDA not available, falling back to CPU")
    device_target = "cpu"

device = torch.device(device_target)
print(f"Device: {device}")
if device.type == "cuda":
    n_gpu = torch.cuda.device_count()
    print(f"Visible GPUs (CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES', 'all')}): {n_gpu}")
    for i in range(n_gpu):
        print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")


---

## Configuration

Load the NARR_PRISM YAML configuration and specify the checkpoint to use.

In [ ]:
from granitewxc.utils.config import get_config
from narr_prism_utils import case_output_dir, get_case_name, load_yaml, resolve_path

# ===================== USER PARAMETERS (EDIT ME) =====================
CONFIG_PATH = NARR_PRISM_DIR / "NARR_PRISM.yaml"
CHECKPOINT_PATH = None   # Set to explicit path, or None for auto-detection
OUTPUT_DIR = None        # Set to override output directory, or None for YAML default
BATCH_SIZE = 8           # Tiles per forward pass per worker GPU
PARALLEL_GPUS = "0,1"    # Physical GPU ids; set None to run in this notebook kernel
# ====================================================================

config_path = str(CONFIG_PATH.resolve())
cfg = load_yaml(config_path)
config = get_config(config_path)
case_name = get_case_name(cfg)

print(f"Config: {config_path}")
print(f"Case name: {case_name}")
print(f"Target variables: {cfg['data']['target_variables']}")
print(f"Inference dates: {cfg['dates']['inference']['start']} to {cfg['dates']['inference']['end']}")
print(f"Parallel GPUs: {PARALLEL_GPUS}")


---

## Locate Checkpoint

Find the trained model checkpoint. The search order is:
1. Explicit `CHECKPOINT_PATH` (if set above)
2. `inference.checkpoint_path` from the YAML
3. `best.ckpt` or `last.ckpt` under the experiment directory

In [ ]:
from narr_prism_inference import _find_checkpoint

checkpoint_path = _find_checkpoint(cfg, CHECKPOINT_PATH)
print(f"Using checkpoint: {checkpoint_path}")

---

## Load Model

Recreate the model architecture and load the trained weights.

In [ ]:
print("Model loading is handled inside run_inference so the notebook does not keep a duplicate GPU copy alive.")

---

## Run Inference

Execute inference over the date range defined in the YAML configuration.
The output is denormalized to physical units and written to a NetCDF file.

In [ ]:
from narr_prism_inference import run_inference, run_parallel_inference

output_root = OUTPUT_DIR or resolve_path(
    cfg.get("inference", {}).get(
        "output_dir", "./examples/NARR_PRISM/experiments/inference_output"
    )
)
output_dir = str(case_output_dir(output_root, case_name))

if PARALLEL_GPUS:
    gpu_ids = [gpu.strip() for gpu in str(PARALLEL_GPUS).split(",") if gpu.strip()]
    output_path = run_parallel_inference(
        config_path=config_path,
        checkpoint_path=checkpoint_path,
        output_dir=output_dir,
        gpu_ids=gpu_ids,
        batch_size=BATCH_SIZE,
        device=device_target,
    )
else:
    output_path = run_inference(
        config_path=config_path,
        cfg=cfg,
        config=config,
        checkpoint_path=checkpoint_path,
        output_dir=output_dir,
        device=device,
        batch_size=BATCH_SIZE,
    )

print(f"\nInference output saved to: {output_path}")


---

## Visualize Results

Load the output NetCDF and plot sample predictions for each target variable.

In [ ]:
from pathlib import Path
import xarray as xr

output_path = Path(output_path)
nc_files = sorted(output_path.glob(f"{case_name}_inference_*.nc"))
if not nc_files:
    raise FileNotFoundError(f"No daily inference NetCDF files found in {output_path}")

try:
    # Preferred path when dask is available: keep the daily files lazily combined.
    ds = xr.open_mfdataset([str(p) for p in nc_files], combine="by_coords")
except (ImportError, ValueError) as exc:
    if len(nc_files) <= 512:
        # Dependency-light fallback for small inference runs.
        daily = [xr.open_dataset(str(p)) for p in nc_files]
        ds = xr.concat(daily, dim="time")
    else:
        print(
            "Install dask to lazily open all daily files with xr.open_mfdataset; "
            "showing the first daily file for inspection."
        )
        print(f"open_mfdataset error: {exc}")
        ds = xr.open_dataset(str(nc_files[0]))

print(f"Daily files: {len(nc_files)}")
print(ds)
print(f"\nTime steps available: {len(nc_files)}")
print(f"Time steps opened in ds: {ds.sizes.get('time', 'n/a')}")
print(f"Spatial grid: {ds.sizes.get('lat', 'n/a')} x {ds.sizes.get('lon', 'n/a')}")

In [ ]:
import matplotlib.pyplot as plt

target_variables = cfg["data"]["target_variables"]
n_vars = len(target_variables)

fig, axes = plt.subplots(1, n_vars, figsize=(6 * n_vars, 5))
if n_vars == 1:
    axes = [axes]

# Plot the first time step for each variable
for ax, var in zip(axes, target_variables):
    data = ds[var].isel(time=0)
    im = ax.pcolormesh(ds.lon, ds.lat, data.values, shading="auto", cmap="coolwarm")
    ax.set_title(f"{var} (t=0)")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle("NARR-to-PRISM Downscaling Predictions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### Spatial Mean Time Series

Plot the spatial-mean time series for each variable to check temporal consistency.

In [ ]:
fig, axes = plt.subplots(n_vars, 1, figsize=(10, 4 * n_vars), sharex=True)
if n_vars == 1:
    axes = [axes]

preview_steps = min(7, len(ds.time))
preview_time = ds.time.isel(time=slice(0, preview_steps)).values

for ax, var in zip(axes, target_variables):
    spatial_mean = []
    for idx in range(preview_steps):
        spatial_mean.append(float(ds[var].isel(time=idx).mean(dim=["lat", "lon"]).values))
    ax.plot(preview_time, spatial_mean, linewidth=0.8, marker="o", markersize=3)
    ax.set_ylabel(var)
    ax.set_title(f"Spatial-mean {var}")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time")
plt.tight_layout()
plt.show()

In [ ]:
ds.close()
print("Done.")

---

## Summary

The inference pipeline:
1. Loaded the trained checkpoint from the case-specific checkpoint directory
2. Ran predictions over the inference date range
3. Wrote physical-unit outputs with proper coordinates and metadata
4. Saved daily NetCDF files under the case-specific inference output directory

Output file location:
```
examples/NARR_PRISM/experiments/inference_output/<case_name>/
```
